In [1]:
import numpy as np

# BatchNorm
Function to implement batch norm

In [ ]:
class BatchNorm2d():
    def __init__(self, channels, training = True):
        # input: [N, C, H, W]
        self.running_mean = np.zeros((1, channels, 1, 1))
        self.running_var = np.zeros((1, channels, 1, 1))
        self.gamma = np.ones((1, channels, 1, 1))
        self.beta = np.zeros((1, channels, 1, 1))

        self.eps = 1e-5
        self.momentum = 0.9
        self.training = training
    
    def __call__(self, input):
        # input: [N, C, H, W]
        assert len(input.shape) == 4
        N, C, H, W = input.shape

        if self.training:
            mean = input.mean(axis=(0, 2, 3), keepdims=True) # [1, C, 1, 1]
            var = input.var(axis=(0, 2, 3), keepdims=True) # [1, C, 1, 1]
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * mean
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * var
        else:
            mean = self.running_mean
            var = self.running_var
        
        output = (input - mean) / np.sqrt(var + self.eps)
        output = output * self.gamma + self.beta
        return output
    
    def eval(self):
        self.training = False
    
    def train(self):
        self.training = True

model = BatchNorm2d(3)
input = np.random.randn(2, 3, 4, 4)
output = model(input)
print(output.shape)
print(model.running_mean.shape, model.running_var.shape)
print(np.linalg.norm(input - output))

# LayernNorm
Function to compute layer norm for input of size (B, T, C) uusually found in language modeling problems

In [ ]:
class LayerNorm1d():
    def __init__(self, features):
        # input: [B, T, C]
        self.gamma = np.ones((features)).astype(np.float32)
        self.beta = np.zeros((features)).astype(np.float32)
        self.eps = 1e-5
    
    def __call__(self, input):
        # input: [B, T, C]
        assert len(input.shape) == 3

        mean = input.mean(axis=(-1), keepdims=True) # [B, T, 1]
        var = input.var(axis=(-1), keepdims=True)
        output = (input - mean) / np.sqrt(var + self.eps)
        output = output * self.gamma + self.beta
        return output


input = np.random.randn(2, 4, 3)
input = input.astype(np.float32)

model = LayerNorm1d(3)
output = model(input)


import torch
model2 = torch.nn.LayerNorm(3)
model2.eval()
model2.weight.data.fill_(1)
model2.bias.data.fill_(0)
output2 = model2(torch.tensor(input, dtype=torch.float32))

# print(output, output2)
print(np.allclose(output, output2.detach().numpy()))

True


![alt text](https://user-images.githubusercontent.com/76557164/176494819-d7be7cd7-a181-417b-aeee-b197c86a2a18.png)